## Correctness Prediction Baselines (Table: Correctness prediction under 5-fold stratified CV)

This notebook reproduces the six feature-set conditions reported in the paper:

| Feature Set | Components | #Features |
|-------------|------------|-----------|
| **Length** | CoT total tokens | 1 |
| **Length + reasoning** | total tokens + thinking-token count + ratio | 3 |
| **Self-revision** | lexical self-revision marker statistics | 3 |
| **ThinkARM** | episode token intensities (Read–Answer) | 8 |
| **SHAPE (H)** | heuristic unit intensities H1–H11 | 11 |
| **SHAPE (H+N)** | heuristic unit intensities H1–H11 + N | 12 |

In [14]:
import os
import json
import re
import math
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import tiktoken
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
print("Imports OK")


Imports OK


## Configuration

Heuristic taxonomy: **H1–H11 + N = 12 labels**.
H sub-type suffixes are stripped to parent (H3a → H3, H11e → H11).
All non-heuristic labels (N1, N2, N3, N4, …) are collapsed to a single **N**.


In [15]:
# Paths resolved relative to the notebook directory (assumes CWD = notebook dir, default in Jupyter/VS Code)
_NB_DIR = os.path.abspath('')
THINKARM_BASE = os.path.normpath(os.path.join(_NB_DIR, "../../ThinkARM"))
SHAPE_BASE    = os.path.normpath(os.path.join(_NB_DIR, "../../math-skills-llm/data/llm-label-semantic-space/thinkarm"))
SHAPE_LABELER = "Qwen-Qwen3.5-27B"
RAW_BASE      = os.path.normpath(os.path.join(_NB_DIR, "../../math-skills-llm/data/raw/thinkarm"))

def _discover_models() -> list[str]:
    shape_models = {
        m for m in os.listdir(SHAPE_BASE)
        if os.path.isdir(os.path.join(SHAPE_BASE, m, SHAPE_LABELER))
    }
    ta_label_dir = os.path.join(THINKARM_BASE, "data", "label")
    ta_models    = {m for m in os.listdir(ta_label_dir)
                    if os.path.isdir(os.path.join(ta_label_dir, m))}
    corr_dir     = os.path.join(THINKARM_BASE, "data", "correct")
    corr_models  = {f[:-5] for f in os.listdir(corr_dir) if f.endswith(".json")}
    available    = sorted(shape_models & ta_models & corr_models)
    skipped      = (shape_models | ta_models) - set(available)
    if skipped:
        print(f"  [skip] missing >=1 source: {sorted(skipped)}")
    return available


# Model configuration
COT_AVAILABLE_MODELS = {
    "DeepSeek-R1-Distill-Qwen-1.5B",
    "DeepSeek-R1-Distill-Qwen-7B",
    "Phi4R",
    "Qwen3_32B",
    "QwQ32B",
    "deepseekR1",
}

INSTRUCT_FINAL_MODELS = {
    "Phi4",
    "Qwen25_32B",
    "gemini2.5flash",
    "gpt4o",
    "o1mini",
    "o3mini",
    "Qwen3_32BNR",
    "gemini2.0flash",
}

MODEL_ALIASES = {
    "deepseekR1": "deepseekR1",
    "DeepSeek-R1": "deepseekR1",
    "DeepSeekR1": "deepseekR1",
    "Qwen3_32B": "Qwen3_32B",
    "Qwen3-32B": "Qwen3_32B",
    "Qwen3_32BNR": "Qwen3_32BNR",
    "Qwen3-32B-NR": "Qwen3_32BNR",
    "Qwen25_32B": "Qwen25_32B",
    "Qwen2.5_32B": "Qwen25_32B",
    "Qwen2.5-32B": "Qwen25_32B",
    "QwQ32B": "QwQ32B",
    "Phi4": "Phi4",
    "Phi4R": "Phi4R",
    "gpt4o": "gpt4o",
    "gemini2.5flash": "gemini2.5flash",
    "gemini2.0flash": "gemini2.0flash",
    "o1mini": "o1mini",
    "o3mini": "o3mini",
}

def canonical_model_name(name: str) -> str:
    return MODEL_ALIASES.get(name, name)

def model_group(name: str) -> str:
    cname = canonical_model_name(name)
    if cname in COT_AVAILABLE_MODELS:
        return "CoT available"
    return "Others"

MODELS = _discover_models()
print(f"Models ({len(MODELS)}): {MODELS}")
print(f"Configured CoT models ({len(COT_AVAILABLE_MODELS)}): {sorted(COT_AVAILABLE_MODELS)}")
print(f"Configured Instruct/final models ({len(INSTRUCT_FINAL_MODELS)}): {sorted(INSTRUCT_FINAL_MODELS)}")

# Episode taxonomy (ThinkARM)
EPISODES = ["Read", "Analyze", "Plan", "Implement",
            "Explore", "Verify", "Monitor", "Answer"]
EP_IDX = {e: i for i, e in enumerate(EPISODES)}
N_EP   = len(EPISODES)   # 8

# Heuristic taxonomy (SHAPE): H1-H11 + N = 12
TAXONOMY = ["H1", "H2", "H3", "H4", "H5", "H6", "H7",
            "H8", "H9", "H10", "H11", "N"]
H_IDX = {h: i for i, h in enumerate(TAXONOMY)}
N_H   = len(TAXONOMY)   # 12

_H_NORM_RE = re.compile(r'^(H\d+)[a-z].*$')
_N_RE      = re.compile(r'^N')

def normalize_tag(raw_tag: str) -> str:
    '''H sub-types → parent (H3a->H3); any N* → N.'''
    m = _H_NORM_RE.match(raw_tag)
    if m: return m.group(1)
    if _N_RE.match(raw_tag): return "N"
    return raw_tag

def norm_tags(lst: list) -> list:
    return [t for t in (normalize_tag(h) for h in lst) if t in H_IDX]

_ENC = tiktoken.get_encoding("cl100k_base")
def token_count(text: str) -> int:
    return len(_ENC.encode(text))

print(f"\nEpisodes  ({N_EP}): {EPISODES}")
print(f"Heuristics({N_H}): {TAXONOMY}")
print(f"Tiktoken  : cl100k_base ready")

Models (15): ['DSQwen32B', 'DeepSeek-R1-Distill-Qwen-1.5B', 'DeepSeek-R1-Distill-Qwen-7B', 'Phi4', 'Phi4R', 'QwQ32B', 'Qwen25_32B', 'Qwen3_32B', 'Qwen3_32BNR', 'deepseekR1', 'gemini2.0flash', 'gemini2.5flash', 'gpt4o', 'o1mini', 'o3mini']
Configured CoT models (6): ['DeepSeek-R1-Distill-Qwen-1.5B', 'DeepSeek-R1-Distill-Qwen-7B', 'Phi4R', 'QwQ32B', 'Qwen3_32B', 'deepseekR1']
Configured Instruct/final models (8): ['Phi4', 'Qwen25_32B', 'Qwen3_32BNR', 'gemini2.0flash', 'gemini2.5flash', 'gpt4o', 'o1mini', 'o3mini']

Episodes  (8): ['Read', 'Analyze', 'Plan', 'Implement', 'Explore', 'Verify', 'Monitor', 'Answer']
Heuristics(12): ['H1', 'H2', 'H3', 'H4', 'H5', 'H6', 'H7', 'H8', 'H9', 'H10', 'H11', 'N']
Tiktoken  : cl100k_base ready


## Section 1: Data Loading

Join ThinkARM labels, SHAPE labels, and correctness on `(model, problem_id)`.


In [16]:
def load_thinkarm(model: str) -> dict:
    label_dir = os.path.join(THINKARM_BASE, "data", "label", model)
    result = {}
    if not os.path.isdir(label_dir):
        return result
    for fname in os.listdir(label_dir):
        if fname.endswith(".json"):
            with open(os.path.join(label_dir, fname), "r", encoding="utf-8") as f:
                result[fname[:-5]] = json.load(f)
    return result

def load_shape(model: str) -> dict:
    shape_dir = os.path.join(SHAPE_BASE, model, SHAPE_LABELER)
    result = {}
    if not os.path.isdir(shape_dir):
        return result
    for fname in os.listdir(shape_dir):
        if fname.endswith(".json"):
            with open(os.path.join(shape_dir, fname), "r", encoding="utf-8") as f:
                result[fname[:-5]] = json.load(f)
    return result

def load_correctness(model: str) -> dict:
    cfile = os.path.join(THINKARM_BASE, "data", "correct", f"{model}.json")
    if not os.path.isfile(cfile):
        return {}
    with open(cfile, "r", encoding="utf-8") as f:
        return json.load(f)

def resolve_correctness(corr_map: dict, pid: str):
    v = corr_map.get(pid)
    if v is None:
        try:
            v = corr_map.get(str(int(pid)))
        except (ValueError, TypeError):
            pass
    return v


def _norm_pid(x):
    if x is None:
        return None
    sx = str(x).strip()
    if sx == "":
        return None
    try:
        return str(int(float(sx)))
    except (ValueError, TypeError):
        return sx

def load_raw(model: str) -> dict:
    """Load raw generation records keyed by question id (with robust key variants)."""
    rfile = os.path.join(RAW_BASE, f"{model}.json")
    if not os.path.isfile(rfile):
        return {}
    with open(rfile, "r", encoding="utf-8") as f:
        data = json.load(f)

    out = {}

    def _put_keys(pid, rec):
        if pid is None:
            return
        s = str(pid).strip()
        if s:
            out[s] = rec
        npid = _norm_pid(pid)
        if npid is not None:
            out[npid] = rec

    if isinstance(data, list):
        for rec in data:
            if not isinstance(rec, dict):
                continue
            pid = rec.get("Question ID", rec.get("question_id", rec.get("id")))
            _put_keys(pid, rec)
            _put_keys(rec.get("index"), rec)
    elif isinstance(data, dict):
        for k, rec in data.items():
            if isinstance(rec, dict):
                _put_keys(k, rec)
                pid = rec.get("Question ID", rec.get("question_id", rec.get("id")))
                _put_keys(pid, rec)
                _put_keys(rec.get("index"), rec)
    return out


In [17]:
raw_data: list[dict] = []
skip_count   = 0
per_model_ok: dict[str, int] = {}
per_model_skip_raw: dict[str, int] = {}

for model in MODELS:
    ta_data   = load_thinkarm(model)
    sh_data   = load_shape(model)
    corr_data = load_correctness(model)
    raw_map   = load_raw(model)
    joined = 0
    skip_raw = 0

    for pid, ta_units in ta_data.items():
        sh_units = sh_data.get(pid)
        if sh_units is None:
            skip_count += 1
            continue

        correct = resolve_correctness(corr_data, pid)
        if correct is None:
            skip_count += 1
            continue

        pid_candidates = [pid, str(pid), _norm_pid(pid)]
        raw_rec = None
        for k in pid_candidates:
            if k is not None and k in raw_map:
                raw_rec = raw_map[k]
                break

        if raw_rec is None:
            skip_count += 1
            skip_raw += 1
            continue

        raw_data.append({
            "ta_units": ta_units,
            "sh_units": sh_units,
            "raw_rec": raw_rec,
            "correct": int(bool(correct)),
            "model": model,
            "pid": pid,
        })
        joined += 1

    per_model_ok[model] = joined
    per_model_skip_raw[model] = skip_raw

print(f"Total joined : {len(raw_data)}  |  skipped : {skip_count}")
print()
print("Per-model joined counts:")
for m, c in per_model_ok.items():
    print(f"  {m:35s}: {c}")
print()
print("Per-model raw-miss counts:")
for m, c in per_model_skip_raw.items():
    print(f"  {m:35s}: {c}")


Total joined : 1500  |  skipped : 0

Per-model joined counts:
  DSQwen32B                          : 100
  DeepSeek-R1-Distill-Qwen-1.5B      : 100
  DeepSeek-R1-Distill-Qwen-7B        : 100
  Phi4                               : 100
  Phi4R                              : 100
  QwQ32B                             : 100
  Qwen25_32B                         : 100
  Qwen3_32B                          : 100
  Qwen3_32BNR                        : 100
  deepseekR1                         : 100
  gemini2.0flash                     : 100
  gemini2.5flash                     : 100
  gpt4o                              : 100
  o1mini                             : 100
  o3mini                             : 100

Per-model raw-miss counts:
  DSQwen32B                          : 0
  DeepSeek-R1-Distill-Qwen-1.5B      : 0
  DeepSeek-R1-Distill-Qwen-7B        : 0
  Phi4                               : 0
  Phi4R                              : 0
  QwQ32B                             : 0
  Qwen25_32B       

## Section 2: Feature Extraction

| Prefix | Component | Source |
|--------|-----------|--------|
| `len_` | CoT total tokens | SHAPE units |
| `g_` | Reasoning token features: total, thinking count, thinking ratio | raw rationale/response |
| `aha_` | Lexical self-revision marker statistics | SHAPE units |
| `i_ta_` | Episode token intensities (ThinkARM) | ThinkARM tokens |
| `i_sh_` | Heuristic unit intensities H1–H11 + N (SHAPE) | SHAPE units |

In [18]:
def feat_intensity_ta(ta_units: list) -> dict:
    """8 features: token fraction per episode."""
    total = 0
    cat_tok: dict = defaultdict(int)
    for u in ta_units:
        n = token_count(u.get("sentence", ""))
        total += n
        cat = u.get("sentence-category", "Monitor")
        if cat in EP_IDX:
            cat_tok[cat] += n
    safe = total if total > 0 else 1
    return {f"i_ta_{ep}": cat_tok[ep] / safe for ep in EPISODES}

In [19]:
def _unit_text(unit: dict) -> str:
    """Return the best available text field from a labelled unit."""
    for key in ("sentence", "text", "content", "raw_text", "unit", "segment"):
        val = unit.get(key)
        if isinstance(val, str) and val.strip():
            return val
    return ""

AHA_MARKERS = [
    "wait", "aha", "hold on", "recheck", "re-check", "reconsider",
    "verify", "to verify", "let me verify", "let's verify", "let's verify",
    "check", "let me check", "let's check", "let's check",
    "confirm", "let's confirm", "let's confirm",
    "alternatively", "another way", "another approach",
    "no,", "no:", "but let's", "but let's"
]
_AHA_RE = re.compile(
    r"(?i)(" + "|".join(re.escape(m) for m in AHA_MARKERS) + r")"
)

def feat_length(sh_units: list) -> dict:
    """1 feature: total token count (CoT length)."""
    unit_tokens = [token_count(_unit_text(u)) for u in sh_units]
    return {"len_total_tokens": int(sum(unit_tokens))}

def feat_self_revision(sh_units: list) -> dict:
    """3 features: lexical self-revision marker statistics."""
    unit_texts = [_unit_text(u) for u in sh_units]
    total_units = len(unit_texts)
    total_tokens = sum(token_count(t) for t in unit_texts)
    marker_flags = [bool(_AHA_RE.search(t)) for t in unit_texts]
    marker_count = int(sum(marker_flags))
    marker_tokens = sum(token_count(t) for t, flag in zip(unit_texts, marker_flags) if flag)
    safe_tokens = total_tokens if total_tokens > 0 else 1
    return {
        "aha_marker_count": marker_count,
        "aha_marker_token_ratio": marker_tokens / safe_tokens,
        "aha_has_marker": float(marker_count > 0),
    }

def feat_length_reasoning(raw_rec: dict) -> dict:
    """3 features: total tokens, thinking-token count, thinking-token ratio."""
    rationale = ""
    response  = ""
    if isinstance(raw_rec, dict):
        rationale = raw_rec.get("Rationale", raw_rec.get("rationale", "")) or ""
        response  = raw_rec.get("Response",  raw_rec.get("response",  "")) or ""
    think_tokens    = token_count(rationale)
    response_tokens = token_count(response)
    total_tokens    = think_tokens + response_tokens
    safe = total_tokens if total_tokens > 0 else 1
    return {
        "g_total_tokens":        total_tokens,
        "g_thinking_tokens":     think_tokens,
        "g_thinking_token_ratio": think_tokens / safe,
    }

def feat_intensity_sh(sh_units: list) -> dict:
    """12 features: unit fraction per heuristic label (H1–H11 + N)."""
    total = len(sh_units)
    safe  = total if total > 0 else 1
    cnt: dict = defaultdict(int)
    for u in sh_units:
        for h in norm_tags(u.get("ontology_tag", [])):
            cnt[h] += 1
    return {f"i_sh_{h}": cnt[h] / safe for h in TAXONOMY}

In [20]:
print("Extracting features...")
records = []
errors  = 0

for entry in raw_data:
    try:
        row: dict = {}
        row.update(feat_length(entry["sh_units"]))
        row.update(feat_length_reasoning(entry["raw_rec"]))
        row.update(feat_self_revision(entry["sh_units"]))
        row.update(feat_intensity_ta(entry["ta_units"]))
        row.update(feat_intensity_sh(entry["sh_units"]))
        row["correctness"] = entry["correct"]
        row["model_name"]  = entry["model"]
        row["problem_id"]  = entry["pid"]
        records.append(row)
    except Exception:
        errors += 1

if len(records) == 0:
    raise RuntimeError("No records were built. Check raw-data pid matching in Section 1.")

df = pd.DataFrame(records).fillna(0)
df["model_group"] = df["model_name"].apply(model_group)
print(f"Feature matrix : {df.shape}  |  errors : {errors}")
print(f"Correctness rate: {df['correctness'].mean():.3f}")
print()
print("Correctness by model:")
print(df.groupby("model_name")["correctness"]
      .agg(["count", "sum", "mean"])
      .rename(columns={"count": "total", "sum": "correct", "mean": "rate"}))

Extracting features...
Feature matrix : (1500, 31)  |  errors : 0
Correctness rate: 0.442

Correctness by model:
                               total  correct   rate
model_name                                          
DSQwen32B                        100       57 0.5700
DeepSeek-R1-Distill-Qwen-1.5B    100       36 0.3600
DeepSeek-R1-Distill-Qwen-7B      100       49 0.4900
Phi4                             100       29 0.2900
Phi4R                            100       37 0.3700
QwQ32B                           100       63 0.6300
Qwen25_32B                       100       28 0.2800
Qwen3_32B                        100       63 0.6300
Qwen3_32BNR                      100       34 0.3400
deepseekR1                       100       63 0.6300
gemini2.0flash                   100       34 0.3400
gemini2.5flash                   100       67 0.6700
gpt4o                            100       19 0.1900
o1mini                           100       37 0.3700
o3mini                           100   

## Section 3: Feature Group Summary


In [21]:
FEATURE_GROUPS = {
    "length"          : [c for c in df.columns if c.startswith("len_")],
    "length_reasoning": [c for c in df.columns if c.startswith("g_")],
    "self_revision"   : [c for c in df.columns if c.startswith("aha_")],
    "thinkarm"        : [c for c in df.columns if c.startswith("i_ta_")],
    "shape_h"         : [c for c in df.columns if c.startswith("i_sh_") and not c.startswith("i_sh_N")],
    "shape_h_n"       : [c for c in df.columns if c.startswith("i_sh_")],
}

print("Feature group sizes:")
for g, cols in FEATURE_GROUPS.items():
    print(f"  {g:20s}: {len(cols):4d}")

def get_cols(groups: list[str]) -> list[str]:
    return [c for g in groups for c in FEATURE_GROUPS[g]]

Feature group sizes:
  length              :    1
  length_reasoning    :    3
  self_revision       :    3
  thinkarm            :    8
  shape_h             :   11
  shape_h_n           :   12


## Section 4: Experimental Conditions

Six conditions matching the paper table, each evaluated with L1 and L2 regularisation (best AUROC is reported):

| Label | Feature group(s) | #Features |
|-------|-----------------|-----------|
| Length | `length` | 1 |
| Length + reasoning | `length_reasoning` | 3 |
| Self-revision | `self_revision` | 3 |
| ThinkARM | `thinkarm` | 8 |
| SHAPE (H) | `shape_h` | 11 |
| SHAPE (H+N) | `shape_h_n` | 12 |

In [22]:
EXPERIMENTS: list[dict] = [
    {"label": "Length",             "groups": ["length"],           "penalty": "l1", "solver": "liblinear"},
    {"label": "Length",             "groups": ["length"],           "penalty": "l2", "solver": "lbfgs"},
    {"label": "Length + reasoning", "groups": ["length_reasoning"], "penalty": "l1", "solver": "liblinear"},
    {"label": "Length + reasoning", "groups": ["length_reasoning"], "penalty": "l2", "solver": "lbfgs"},
    {"label": "Self-revision",      "groups": ["self_revision"],    "penalty": "l1", "solver": "liblinear"},
    {"label": "Self-revision",      "groups": ["self_revision"],    "penalty": "l2", "solver": "lbfgs"},
    {"label": "ThinkARM",           "groups": ["thinkarm"],         "penalty": "l1", "solver": "liblinear"},
    {"label": "ThinkARM",           "groups": ["thinkarm"],         "penalty": "l2", "solver": "lbfgs"},
    {"label": "SHAPE (H)",          "groups": ["shape_h"],          "penalty": "l1", "solver": "liblinear"},
    {"label": "SHAPE (H)",          "groups": ["shape_h"],          "penalty": "l2", "solver": "lbfgs"},
    {"label": "SHAPE (H+N)",        "groups": ["shape_h_n"],        "penalty": "l1", "solver": "liblinear"},
    {"label": "SHAPE (H+N)",        "groups": ["shape_h_n"],        "penalty": "l2", "solver": "lbfgs"},
]

## Section 5: Training & Evaluation

**Protocol**: 5-fold stratified cross-validation on the full dataset.
**Scaling**: `StandardScaler` fit on train fold only, applied to test fold.
**Inner CV**: 5-fold AUROC to select regularisation strength C ∈ {0.01, 0.05, 0.1, 0.5, 1.0, 5.0}.


In [23]:
def evaluate_experiment(
    df: pd.DataFrame,
    feature_cols: list[str],
    penalty: str,
    solver: str,
    n_splits: int = 5,
) -> tuple[pd.DataFrame, list[float]]:
    """Stratified 5-fold CV for one (feature_set, regularisation) combination."""
    from sklearn.model_selection import StratifiedKFold

    X = df[feature_cols].values.astype(float)
    y = df["correctness"].values
    split_labels = df["model_group"].values

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_rows: list = []
    best_Cs  : list = []

    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_te, y_te = X[te_idx], y[te_idx]
        te_split   = split_labels[te_idx]

        if len(np.unique(y_tr)) < 2 or len(np.unique(y_te)) < 2:
            continue

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

        clf = LogisticRegressionCV(
            penalty=penalty, solver=solver,
            Cs=[0.01, 0.05, 0.1, 0.5, 1.0, 5.0],
            cv=5, scoring="roc_auc",
            random_state=42, max_iter=2000,
        )
        clf.fit(X_tr_s, y_tr)
        best_Cs.append(float(clf.C_[0]))

        proba = clf.predict_proba(X_te_s)[:, 1]

        def _metric_row(mask, split_name):
            if mask.sum() == 0:
                return None
            yy = y_te[mask]
            pp = proba[mask]
            if len(np.unique(yy)) < 2:
                return None
            return {
                "fold": fold_idx,
                "split": split_name,
                "n_samples": int(mask.sum()),
                "auroc": float(roc_auc_score(yy, pp)),
            }

        rows = [
            _metric_row(np.ones_like(y_te, dtype=bool), "ALL"),
            _metric_row(te_split == "CoT available", "CoT available"),
            _metric_row(te_split == "Others", "Others"),
        ]
        for r in rows:
            if r is not None:
                fold_rows.append(r)

    return pd.DataFrame(fold_rows), best_Cs

In [24]:
all_folds  : list[pd.DataFrame] = []
all_best_Cs: list[list]         = []

for i, exp in enumerate(EXPERIMENTS):
    fcols = get_cols(exp["groups"])
    folds, best_Cs = evaluate_experiment(df, fcols, exp["penalty"], exp["solver"])
    all_folds.append(folds)
    all_best_Cs.append(best_Cs)
    tag = f"{exp['label']:20s} / {exp['penalty'].upper()}"

    fd_all = folds[folds["split"] == "ALL"] if len(folds) > 0 else pd.DataFrame()
    if len(fd_all) > 0:
        print(f"  {i+1:2d}. {tag}: AUROC = {fd_all['auroc'].mean():.4f} ± {fd_all['auroc'].std():.4f}")
    else:
        print(f"  {i+1:2d}. {tag}: no valid folds")

# Per-experiment AUROC summary (ALL split only)
_num_exp: list = []
for i in range(len(EXPERIMENTS)):
    fd = all_folds[i]
    fd_all = fd[fd["split"] == "ALL"] if len(fd) > 0 else pd.DataFrame()
    if len(fd_all) == 0:
        _num_exp.append(None)
    else:
        _num_exp.append({"auroc": fd_all["auroc"].mean()})

# Group-wise AUROC aggregation
_group_rows = []
for i, exp in enumerate(EXPERIMENTS):
    fd = all_folds[i]
    if len(fd) == 0:
        continue
    for split_name in ["CoT available", "Others", "ALL"]:
        sub = fd[fd["split"] == split_name]
        if len(sub) == 0:
            continue
        _group_rows.append({
            "exp_idx": i,
            "Condition": exp["label"],
            "Penalty": exp["penalty"].upper(),
            "#Features": len(get_cols(exp["groups"])),
            "Split": split_name,
            "#Folds": int(sub["fold"].nunique()),
            "AUROC": float(sub["auroc"].mean()),
        })

exp_group_metrics_df = pd.DataFrame(_group_rows)
print("Built group-wise metrics table:", exp_group_metrics_df.shape)

   1. Length               / L1: AUROC = 0.4990 ± 0.0257
   2. Length               / L2: AUROC = 0.5039 ± 0.0282
   3. Length + reasoning   / L1: AUROC = 0.4985 ± 0.0255
   4. Length + reasoning   / L2: AUROC = 0.5034 ± 0.0282
   5. Self-revision        / L1: AUROC = 0.6153 ± 0.0285
   6. Self-revision        / L2: AUROC = 0.6176 ± 0.0312
   7. ThinkARM             / L1: AUROC = 0.6183 ± 0.0244
   8. ThinkARM             / L2: AUROC = 0.6175 ± 0.0243
   9. SHAPE (H)            / L1: AUROC = 0.6525 ± 0.0184
  10. SHAPE (H)            / L2: AUROC = 0.6524 ± 0.0182
  11. SHAPE (H+N)          / L1: AUROC = 0.6641 ± 0.0158
  12. SHAPE (H+N)          / L2: AUROC = 0.6641 ± 0.0159
Built group-wise metrics table: (36, 7)



## Section 6: Summary Table

Mean ± std across 5-fold CV (ALL split). **Bold** = best per column.


In [25]:
display_rows = []
_row_nums    = []

for exp_i, exp in enumerate(EXPERIMENTS):
    fd     = all_folds[exp_i]
    fd_all = fd[fd["split"] == "ALL"] if len(fd) > 0 else pd.DataFrame()
    tag    = f"{exp['label']} / {exp['penalty'].upper()}"
    if len(fd_all) == 0:
        display_rows.append({"Condition / Reg.": tag, "AUROC ↑": "—"})
        _row_nums.append(None)
    else:
        n = _num_exp[exp_i]
        _row_nums.append(n)
        display_rows.append({
            "Condition / Reg.": tag,
            "AUROC ↑": f"{n['auroc']:.4f} ± {fd_all['auroc'].std():.4f}",
        })

sum_df = pd.DataFrame(display_rows)

valid = [n for n in _row_nums if n]
best_auroc = max(n["auroc"] for n in valid) if valid else None

def _bold_row(row):
    styles   = [""] * len(row)
    col_list = list(row.index)
    n = _row_nums[row.name]
    if n is None or best_auroc is None:
        return styles
    if n.get("auroc") == best_auroc:
        styles[col_list.index("AUROC ↑")] = "font-weight: bold"
    return styles

display(sum_df.style
        .apply(_bold_row, axis=1)
        .hide(axis="index")
        .set_caption("5-fold CV AUROC by regularisation (ALL split, mean ± std)"))

Condition / Reg.,AUROC ↑
Length / L1,0.4990 ± 0.0257
Length / L2,0.5039 ± 0.0282
Length + reasoning / L1,0.4985 ± 0.0255
Length + reasoning / L2,0.5034 ± 0.0282
Self-revision / L1,0.6153 ± 0.0285
Self-revision / L2,0.6176 ± 0.0312
ThinkARM / L1,0.6183 ± 0.0244
ThinkARM / L2,0.6175 ± 0.0243
SHAPE (H) / L1,0.6525 ± 0.0184
SHAPE (H) / L2,0.6524 ± 0.0182


In [27]:
def _best_auroc_idx(indices):
    return max(indices, key=lambda i: _num_exp[i]["auroc"] if _num_exp[i] else float("-inf"))

def _idxs(label: str) -> list[int]:
    return [i for i, exp in enumerate(EXPERIMENTS) if exp["label"] == label]

focus_conditions = [
    "Length",
    "Length + reasoning",
    "Self-revision",
    "ThinkARM",
    "SHAPE (H)",
    "SHAPE (H+N)",
]

rows = []
for cond in focus_conditions:
    idx = _best_auroc_idx(_idxs(cond))
    n_feats = len(get_cols(EXPERIMENTS[idx]["groups"]))

    fd_all = all_folds[idx][all_folds[idx]["split"] == "ALL"]
    if len(fd_all) == 0:
        continue
    n = _num_exp[idx]
    rows.append({
        "Feature Set": cond,
        "#Features": n_feats,
        f"AUROC ↑": f"{n['auroc']:.3f}±{fd_all['auroc'].std():.2f}",
    })

paper_df = pd.DataFrame(rows)
display(paper_df.style.hide(axis="index").set_caption(
    "Table: Correctness prediction under 5-fold stratified CV"
))

Feature Set,#Features,AUROC ↑
Length,1,0.504±0.03
Length + reasoning,3,0.503±0.03
Self-revision,3,0.618±0.03
ThinkARM,8,0.618±0.02
SHAPE (H),11,0.653±0.02
SHAPE (H+N),12,0.664±0.02
